# 🚦 Traffic Analyzer – Object Detection (YOLOv8)

**Name:** <your name>  
**Project type:** Object Detection

## 1. Problem Statement
Manual traffic monitoring is slow and error-prone. This project detects and counts vehicles (cars, buses, trucks, motorcycles) and pedestrians in traffic images/videos to estimate congestion levels.

## 2. Idea & Architecture
```
Input (image / video)
        │
        ▼
Frame extraction (OpenCV)
        │
        ▼
YOLOv8 detector (pretrained on COCO)
   backbone → neck → detection head
        │
        ▼
Filter vehicle classes + count per class
        │
        ▼
Traffic density level (Low / Medium / High)
        │
        ▼
Annotated output + summary chart
```

## 3. Install & Import

In [ ]:
!pip install -q ultralytics
import cv2, os
import matplotlib.pyplot as plt
from collections import Counter
from ultralytics import YOLO
from IPython.display import Image, display, Video

## 4. Load Pretrained Model
YOLOv8 pretrained on the COCO dataset (80 classes, includes car, bus, truck, motorcycle, person).

In [ ]:
model = YOLO('yolov8n.pt')
VEHICLES = {'car', 'bus', 'truck', 'motorcycle', 'bicycle'}
print('Model loaded. Classes:', len(model.names))

## 5. Get a Sample Traffic Image
Run this cell, then upload any traffic image from your computer (or use the sample below).

In [ ]:
from google.colab import files
uploaded = files.upload()
IMG_PATH = list(uploaded.keys())[0]
print('Using:', IMG_PATH)

## 6. Detect Objects

In [ ]:
results = model(IMG_PATH)
annotated = results[0].plot()[:, :, ::-1]   # BGR -> RGB
plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title('Detected objects')
plt.show()

## 7. Count Vehicles & Estimate Congestion

In [ ]:
names = [model.names[int(c)] for c in results[0].boxes.cls]
counts = Counter(names)
vehicle_counts = {k: v for k, v in counts.items() if k in VEHICLES}
total = sum(vehicle_counts.values())

if total < 10:
    level = 'LOW'
elif total < 25:
    level = 'MEDIUM'
else:
    level = 'HIGH'

print('Vehicle counts:', vehicle_counts)
print('Total vehicles:', total)
print('Traffic density:', level)

plt.bar(vehicle_counts.keys(), vehicle_counts.values())
plt.title(f'Vehicle counts  |  Density: {level}')
plt.ylabel('Count')
plt.show()

## 8. (Optional) Run on a Video

In [ ]:
# Upload a short traffic video (.mp4) and run this cell
vid = files.upload()
VIDEO_PATH = list(vid.keys())[0]
model.predict(VIDEO_PATH, save=True, conf=0.4)
# Output is saved in runs/detect/predict*/

In [ ]:
import glob, subprocess
out = sorted(glob.glob('runs/detect/predict*/*.avi') + glob.glob('runs/detect/predict*/*.mp4'))[-1]
subprocess.run(['ffmpeg','-y','-i',out,'result.mp4'], capture_output=True)
Video('result.mp4', embed=True, width=640)

## 9. Results & Conclusion
- The pretrained YOLOv8n model detects vehicles in real time with no extra training.
- Counts per class are used to estimate traffic density.

**Future improvements:** fine-tune on a local traffic dataset, add object tracking (ByteTrack) to count unique vehicles, add speed estimation, and detect traffic violations.